# Add Cybernetic Reaction Sandbox to ATS-flow model xml files
- Input: ATS flow xml files for ATS 1.6 in folder, `caseflow-run1` and `caseflow-run2`
- Output: ATS-PFLOTRAN xml files in the same folders with suffix `*.v1.6_pflotran.xml`
- Notes
    - senario s1
        - BCs of reactive transport: dynamic [NH4+], [NO3-], [DOC] from ELM
        - ICs of reactive transport: average value for [NH4+] and [NO3-]; for [DOC], average DOC injection rate / precipitation provides a better estimate (not tested yet)
    - Timestep Controller Type: "fixed" and put 1000s

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from pathlib import Path
import shutil
import subprocess
import h5py as h5
import matplotlib.pyplot as plt
import re
import glob
import numpy as np

from scipy.io import loadmat

In [ ]:
# a temp solution, need to update lib
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, message='.*product.*')

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
hucs           = [config['hucs']]
site_name      = config['site_name']

# simulation control
start_year_spinup         = config['start_year_spinup']
end_year_spinup           = config['end_year_spinup']
nyears_steadystate_spinup = config['nyears_steadystate_spinup']
nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
start_year_transient      = config['start_year_transient']
end_year_transient        = config['end_year_transient']
run                   = config['elm_run']

flag_scenario = 's1' # choose which scenario
outputs = {}

In [ ]:
# check get_docflux_from_ELM_3D.ipynb for interpretation of f_DOM and k
#f_DOM, fdom = 1, '1'
f_DOM, fdom = 0.01, '001'

# k ranges from 0.1~1.5 h^-1, to convert DOC flux to DOC concentration
k_in_sec = np.array([0.1, 1.5])/3600.0
#k, k_label = k_in_sec[0], '01'
k, k_label = k_in_sec[1], '15'

# years of data used to generate the source term and boundary conditions
# noticing the difference to nyears_steadystate_spinup and nyears_cyclic_spinup loaded from config.json
years_spinup    = np.arange(start_year_spinup, end_year_spinup+1)
years_transient = np.arange(start_year_transient, end_year_transient+1)

print(list(years_spinup))
print(list(years_transient))

In [ ]:
m2_mat_filename =  f'../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
meshsize_nx = loaded_data['meshsize_nx'].flatten()[0]
dzs_soil  = loaded_data['dzs_soil'].flatten()
dzs_geo   = loaded_data['dzs_geo'].flatten()
meshsize_nz = len(dzs_soil) + len(dzs_geo)

In [ ]:
# Add atspflotranutils to sys.path
base_dir = Path().resolve().parent  # Assumes the notebook is inside the notebooks/ directory
utils_path = base_dir / 'atspflotranutils'
sys.path.append(str(utils_path))

# Check if the path was added successfully
print("Paths in sys.path:")
print("\n".join(sys.path))

In [ ]:
# copy the processed ELM data to folder data-processed
source_pattern = base_dir / 'notebooks/ELM_from_huilin' / f'{site_name}_*.h5'
target_path = base_dir / 'data-processed' / f'{site_name}'
# Copy all {site_name}_*.h5 files to data-processed
for source_file in glob.glob(str(source_pattern)):
    shutil.copy(source_file, target_path)
    print(f"Copied: {Path(source_file).name}")

In [ ]:
# For spinup, use
# - NF01_DOC_source_spinup_10yr_2011_2015_fdom001.h5
# - NF01_CNbc_conc_spinup_10yr_2011_2015_fdom001_k15.h5
# For transient, use
# - NF01_DOC_source_transient_2016_2020_fdom001_2024-07-14-142626.h5
# - NF01_CNbc_conc_transient_2016_2020_fdom001_k15_2024-07-14-142626.h5
#spinup_doc = target_path / f'NF01_DOC_source_spinup_10yr_2011_2015_fdom001.h5'
spinup_doc = target_path / f'{site_name}_DOC_source_spinup_{nyears_cyclic_spinup}yr_{years_spinup[0]}_{years_spinup[-1]}_fdom{fdom}.h5'
#spinup_cnbc = target_path / 'NF01_CNbc_conc_spinup_10yr_2011_2015_fdom001_k15.h5'
spinup_cnbc = target_path / f'{site_name}_CNbc_conc_spinup_{nyears_cyclic_spinup}yr_{years_spinup[0]}_{years_spinup[-1]}_fdom{fdom}_k{k_label}.h5'
#transient_doc = target_path / 'NF01_DOC_source_transient_2016_2020_fdom001_2024-07-14-142626.h5'
transient_doc = target_path / f'{site_name}_DOC_source_transient_{years_transient[0]}_{years_transient[-1]}_fdom{fdom}_{run}.h5'
#transient_cnbc = target_path / 'NF01_CNbc_conc_transient_2016_2020_fdom001_k15_2024-07-14-142626.h5'
transient_cnbc = target_path / f'{site_name}_CNbc_conc_transient_{years_transient[0]}_{years_transient[-1]}_fdom{fdom}_k{k_label}_{run}.h5'

In [ ]:
# Load spinup data
print("Loading spinup data...")
spinup_doc_data = {}
f = h5.File(spinup_doc, 'r')
spinup_doc_data['DOC production [molC m^-3 s^-1]'] = [np.array(f['DOC production [molC m^-3 s^-1]'][str(i)], dtype=float) for i in range(len(f['DOC production [molC m^-3 s^-1]']))]
spinup_doc_data['time [s]'] = np.array(f['time [s]'])
spinup_doc_data['x [m]'] = np.array(f['x [m]'])
spinup_doc_data['y [m]'] = np.array(f['y [m]'])
f.close()

spinup_cnbc_data = {}
f = h5.File(spinup_cnbc, 'r')
spinup_cnbc_data['Time'] = np.array(f['Time'])
spinup_cnbc_data['NH4+ bulk volume basis [molS L^-1]'] = np.array(f['NH4+ bulk volume basis [molS L^-1]'])
spinup_cnbc_data['NO3- bulk volume basis [molS L^-1]'] = np.array(f['NO3- bulk volume basis [molS L^-1]'])
spinup_cnbc_data['NH4+ mol water basis [molS molH^-1]'] = np.array(f['NH4+ mol water basis [molS molH^-1]'])
spinup_cnbc_data['NO3- mol water basis [molS molH^-1]'] = np.array(f['NO3- mol water basis [molS molH^-1]'])
spinup_cnbc_data['DOC bulk volume basis v1 [molS L^-1]'] = np.array(f['DOC bulk volume basis v1 [molS L^-1]'])
spinup_cnbc_data['DOC mol water basis v1 [molS molH^-1]'] = np.array(f['DOC mol water basis v1 [molS molH^-1]'])
#spinup_cnbc_data['DOC bulk volume basis v2 [molS L^-1]'] = np.array(f['DOC bulk volume basis v2 [molS L^-1]'])
#spinup_cnbc_data['DOC mol water basis v2 [molS molH^-1]'] = np.array(f['DOC mol water basis v2 [molS molH^-1]'])
f.close()

# Load transient data
print("Loading transient data...")
transient_doc_data = {}
f = h5.File(transient_doc, 'r')
transient_doc_data['DOC production [molC m^-3 s^-1]'] = [np.array(f['DOC production [molC m^-3 s^-1]'][str(i)], dtype=float) for i in range(len(f['DOC production [molC m^-3 s^-1]']))]
transient_doc_data['time [s]'] = np.array(f['time [s]'])
transient_doc_data['x [m]'] = np.array(f['x [m]'])
transient_doc_data['y [m]'] = np.array(f['y [m]'])
f.close()

transient_cnbc_data = {}
f = h5.File(transient_cnbc, 'r')
transient_cnbc_data['Time'] = np.array(f['Time'])
transient_cnbc_data['NH4+ bulk volume basis [molS L^-1]'] = np.array(f['NH4+ bulk volume basis [molS L^-1]'])
transient_cnbc_data['NO3- bulk volume basis [molS L^-1]'] = np.array(f['NO3- bulk volume basis [molS L^-1]'])
transient_cnbc_data['NH4+ mol water basis [molS molH^-1]'] = np.array(f['NH4+ mol water basis [molS molH^-1]'])
transient_cnbc_data['NO3- mol water basis [molS molH^-1]'] = np.array(f['NO3- mol water basis [molS molH^-1]'])
transient_cnbc_data['DOC bulk volume basis v1 [molS L^-1]'] = np.array(f['DOC bulk volume basis v1 [molS L^-1]'])
transient_cnbc_data['DOC mol water basis v1 [molS molH^-1]'] = np.array(f['DOC mol water basis v1 [molS molH^-1]'])
f.close()

# plot DOC injection flux and BC concentrations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute domain-averaged DOC production for spinup
doc_prod_spinup = spinup_doc_data['DOC production [molC m^-3 s^-1]']
time_spinup = spinup_doc_data['time [s]']
x_spinup = spinup_doc_data['x [m]']

# Calculate domain average for each time step
doc_avg_spinup = []
for i in range(len(doc_prod_spinup)):
    # Each element is a spatial array, compute mean over all spatial points
    doc_avg_spinup.append(np.mean(doc_prod_spinup[i]))
doc_avg_spinup = np.array(doc_avg_spinup)

# Convert time to days
time_days_spinup = time_spinup / (86400)

# Compute domain-averaged DOC production for transient
doc_prod_transient = transient_doc_data['DOC production [molC m^-3 s^-1]']
time_transient = transient_doc_data['time [s]']

# Calculate domain average for each time step
doc_avg_transient = []
for i in range(len(doc_prod_transient)):
    doc_avg_transient.append(np.mean(doc_prod_transient[i]))
doc_avg_transient = np.array(doc_avg_transient)

# Convert time to years (shift to start after spinup)
time_days_transient = time_transient / (86400) + nyears_cyclic_spinup*365

# Create the plot
fig, ax = plt.subplots(figsize=(14, 6))

# Plot spinup
ax.plot(time_days_spinup, doc_avg_spinup, linewidth=1.5, color='steelblue', 
        label='Spinup (10-year cyclic: 2011-2015)')

# Plot transient
ax.plot(time_days_transient, doc_avg_transient, linewidth=1.5, color='coral', 
        label='Transient (2016-2020)')

# Add vertical line to mark transition
ax.axvline(x=time_days_spinup[-1], color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(time_days_spinup[-1], ax.get_ylim()[1]*0.95, 'Spinup→Transient', 
        ha='center', va='top', fontsize=10, color='gray')

ax.set_xlabel('Time [years]', fontsize=12)
ax.set_ylabel('Domain-averaged DOC production [molC m$^{-3}$ s$^{-1}$]', fontsize=12)
ax.set_title('Domain-averaged DOC Production: Spinup and Transient', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='best')

plt.tight_layout()
plt.show()

In [ ]:
# Get data for spinup
time_days_spinup = spinup_cnbc_data['Time'] / 86400
doc_v1_spinup = spinup_cnbc_data['DOC mol water basis v1 [molS molH^-1]']
nh4_spinup = spinup_cnbc_data['NH4+ mol water basis [molS molH^-1]']
no3_spinup = spinup_cnbc_data['NO3- mol water basis [molS molH^-1]']

# Get data for transient
time_days_transient = transient_cnbc_data['Time'] / 86400 + nyears_cyclic_spinup*365
doc_v1_transient = transient_cnbc_data['DOC mol water basis v1 [molS molH^-1]']
nh4_transient = transient_cnbc_data['NH4+ mol water basis [molS molH^-1]']
no3_transient = transient_cnbc_data['NO3- mol water basis [molS molH^-1]']

# Create figure with 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: DOC mol water basis v1
ax = axes[0]
ax.plot(time_days_spinup, doc_v1_spinup, linewidth=1.5, color='steelblue',
        label='Spinup (10-year cyclic: 2011-2015)')
ax.plot(time_days_transient, doc_v1_transient, linewidth=1.5, color='coral',
        label='Transient (2016-2020)')
ax.axvline(x=time_days_spinup[-1], color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(time_days_spinup[-1], ax.get_ylim()[1]*0.95, 'Spinup→Transient',
        ha='center', va='top', fontsize=10, color='gray')
ax.set_xlabel('Time [days]', fontsize=12)
ax.set_ylabel('DOC v1 [molS molH$^{-1}$]', fontsize=12)
ax.set_title('DOC mol water basis v1', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=10, loc='best')

# Plot 2: NH4+ mol water basis
ax = axes[1]
ax.plot(time_days_spinup, nh4_spinup, linewidth=1.5, color='steelblue',
        label='Spinup (10-year cyclic: 2011-2015)')
ax.plot(time_days_transient, nh4_transient, linewidth=1.5, color='coral',
        label='Transient (2016-2020)')
ax.axvline(x=time_days_spinup[-1], color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(time_days_spinup[-1], ax.get_ylim()[1]*0.95, 'Spinup→Transient',
        ha='center', va='top', fontsize=10, color='gray')
ax.set_xlabel('Time [days]', fontsize=12)
ax.set_ylabel('NH4$^+$ [molS molH$^{-1}$]', fontsize=12)
ax.set_title('NH4+ mol water basis', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=10, loc='best')

# Plot 3: NO3- mol water basis
ax = axes[2]
ax.plot(time_days_spinup, no3_spinup, linewidth=1.5, color='steelblue',
        label='Spinup (10-year cyclic: 2011-2015)')
ax.plot(time_days_transient, no3_transient, linewidth=1.5, color='coral',
        label='Transient (2016-2020)')
ax.axvline(x=time_days_spinup[-1], color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(time_days_spinup[-1], ax.get_ylim()[1]*0.95, 'Spinup→Transient',
        ha='center', va='top', fontsize=10, color='gray')
ax.set_xlabel('Time [days]', fontsize=12)
ax.set_ylabel('NO3$^-$ [molS molH$^{-1}$]', fontsize=12)
ax.set_title('NO3- mol water basis', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=10, loc='best')

plt.tight_layout()
plt.show()

## Merge and Create hdf5 files

In [ ]:
# Define output directory and new filenames
output_dir = target_path

merged_doc_filename = output_dir / f'{site_name}_DOC_source_merged_spinup{nyears_cyclic_spinup}yr_transient_{start_year_transient}_{end_year_transient}_fdom{fdom}_{run}.h5'
merged_cnbc_filename = output_dir / f'{site_name}_CNbc_conc_merged_spinup{nyears_cyclic_spinup}yr_transient_{start_year_transient}_{end_year_transient}_fdom{fdom}_{run}.h5'

In [ ]:
# --- 1. Merge CNbc Concentration Data ---
print(f"Merging CNbc concentration data into:\n  {merged_cnbc_filename}")

with h5.File(merged_cnbc_filename, 'w') as hdf:
    # --- Time Handling ---
    spinup_time = spinup_cnbc_data['Time']
    transient_time = transient_cnbc_data['Time']
    
    # Calculate the time step (usually 86400 seconds/day)
    time_step = spinup_time[1] - spinup_time[0]
    
    # Calculate the offset to add to the transient time
    time_offset = spinup_time[-1] + time_step
    
    # Create the merged time array
    merged_time = np.concatenate((spinup_time, transient_time + time_offset))
    hdf.create_dataset('Time', data=merged_time)
    print("  - Merged 'Time'")

    # --- Merge Other Variables ---
    # Loop through keys (excluding 'Time' which is already done)
    for key in spinup_cnbc_data.keys():
        if key != 'Time':
            spinup_array = spinup_cnbc_data[key]
            transient_array = transient_cnbc_data[key]
            
            # Concatenate the arrays
            merged_array = np.concatenate((spinup_array, transient_array))
            
            # Write to the new HDF5 file
            hdf.create_dataset(key, data=merged_array)
            print(f"  - Merged '{key}'")

print("✓ CNbc merge complete.\n")


# --- 2. Merge DOC Source Data ---
print(f"Merging DOC source data into:\n  {merged_doc_filename}")

with h5.File(merged_doc_filename, 'w') as hdf:
    # --- Time Handling (same logic as before) ---
    spinup_time = spinup_doc_data['time [s]']
    transient_time = transient_doc_data['time [s]']
    time_step = spinup_time[1] - spinup_time[0]
    time_offset = spinup_time[-1] + time_step
    merged_time = np.concatenate((spinup_time, transient_time + time_offset))
    hdf.create_dataset('time [s]', data=merged_time)
    print("  - Merged 'time [s]'")
    
    # --- Copy Static Spatial Data ---
    hdf.create_dataset('x [m]', data=spinup_doc_data['x [m]'])
    hdf.create_dataset('y [m]', data=spinup_doc_data['y [m]'])
    print("  - Copied 'x [m]' and 'y [m]'")

    # --- Special Handling for DOC Production (list of arrays) ---
    doc_prod_key = 'DOC production [molC m^-3 s^-1]'
    spinup_list = spinup_doc_data[doc_prod_key]
    transient_list = transient_doc_data[doc_prod_key]
    print(type(spinup_list))
    print(type(transient_list))
    
    # Combine the two lists
    merged_doc_list = spinup_list + transient_list
    
    # Re-create the group structure from the original file
    doc_prod_group = hdf.create_group(doc_prod_key)
    for i, array_data in enumerate(merged_doc_list):
        doc_prod_group.create_dataset(str(i), data=array_data)
    print(f"  - Merged '{doc_prod_key}' (as a group)")

# pflotranate atsflow xml to atspflotran xml

In [ ]:
from pflotranate_2d import pflotranate
# from pflotranate_3d import pflotranate
# from pflotranate_2dtracers import pflotranate
pflotranate_path = base_dir / 'atspflotranutils' / 'pflotranate_2d'

import importlib
importlib.reload(pflotranate)

In [ ]:
# Prepare '../casecybernetic-run{1,2}' folders
source_path_run1 = base_dir / 'caseflow-run1' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run1.v1.6.xml'
target_folder_run1 = base_dir / f'casecybernetic-run1.{flag_scenario}'
target_path_run1 = target_folder_run1 / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run1.v1.6.xml'
os.makedirs(target_folder_run1, exist_ok=True)
shutil.copy(source_path_run1, target_path_run1)

source_path_run2 = base_dir / 'caseflow-run2' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run2.v1.6.xml'
target_folder_run2 = base_dir / f'casecybernetic-run2.{flag_scenario}'
target_path_run2 = target_folder_run2 / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run2.v1.6.xml'
os.makedirs(target_folder_run2, exist_ok=True)
shutil.copy(source_path_run2, target_path_run2)

## copy corresponding folder-reactions to case folders
source_path_reactions = pflotranate_path / 'reactions'
try:
    shutil.copytree(source_path_reactions, target_folder_run1 / 'reactions', dirs_exist_ok=True)
    shutil.copytree(source_path_reactions, target_folder_run2 / 'reactions', dirs_exist_ok=True)
except FileExistsError as e:
    print(f"Target folder already exists: {e}")

In [ ]:
# Simulate command-line arguments
command = f"python {pflotranate_path / 'pflotranate.py'} {target_path_run1}"

try:
    subprocess.run(command, shell=True, check=True)
    #print(f"Successfully executed: {command}")
except subprocess.CalledProcessError as e:
    print(f"Error occurred: {e}")

# modify the xml file for casecybernetic-run1

In [ ]:
# amanzi_xml, included in AMANZI_SRC_DIR/tools/amanzi_xml
import amanzi_xml.utils.io as aio
import amanzi_xml.utils.search as asearch
import amanzi_xml.utils.errors as aerrors
from amanzi_xml.common.parameter import Parameter
from amanzi_xml.common.parameter_list import ParameterList

In [ ]:
xml_filename = base_dir / f'casecybernetic-run1.{flag_scenario}' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run1.v1.6_pflotran.xml'
xml = aio.fromFile(xml_filename)

## revise the initial conditions for flow part

In [ ]:
caseflow_run1_checkpoint = f'../../caseflow-run1/{site_name}/checkpoint_final.h5'

subsurface_flow_IC = asearch.find_path(xml, ['PKs', 'subsurface flow', 'initial conditions', 'restart file'])
subsurface_flow_IC.set("value", caseflow_run1_checkpoint)
print(subsurface_flow_IC)
aio.toFile(xml, xml_filename)

## revise the soil domain info used for DOC injection source terms

In [ ]:
#soil_domain_region = "{NRCS-68932, NRCS-68901}" # copied from last section in 1-main_workflow_OakCreek.NF01.ats1.5.ipynb
outputs['soil_region_string'] = f'../data-processed/{site_name}/soil_region.txt'
with open(outputs['soil_region_string'], 'r') as f:
    loaded_string = f.read().strip()
soil_domain_region = f"{{{loaded_string}}}"

subsurface_mass_source = asearch.find_path(xml, ['PKs', 'source terms', 'component mass source', 'DOC production', 'regions'])
subsurface_mass_source.set("value", soil_domain_region)
print(subsurface_mass_source)
aio.toFile(xml, xml_filename)

## revise the DOC injection hdf5 file info

In [ ]:
## for ats-pflotran spinup starts from t=0s, it's still okay to use the spinup DOC_source and CNBC h5file
#elm_filename = "../../data-processed/NF01/NF01_DOC_source_spinup_10yr_2011_2015_fdom001.h5"
elm_filename = f'../../data-processed/{site_name}/{site_name}_DOC_source_spinup_{nyears_cyclic_spinup}yr_{years_spinup[0]}_{years_spinup[-1]}_fdom{fdom}.h5'

subsurface_mass_source = asearch.find_path(xml, ['PKs', 'source terms', 'component mass source', 'DOC production', 'source function', 'file'])
subsurface_mass_source.set("value", elm_filename)
print(subsurface_mass_source)
aio.toFile(xml, xml_filename)

## revise the CN boundary conditions hdf5 file info

In [ ]:
## for ats-pflotran spinup starts from t=0s, it's still okay to use the spinup DOC_source and CNBC h5file
#elm_filename = "../../data-processed/NF01/NF01_DOC_source_spinup_10yr_2011_2015_fdom001.h5"
elm_filename = f'../../data-processed/{site_name}/{site_name}_CNbc_conc_spinup_{nyears_cyclic_spinup}yr_{years_spinup[0]}_{years_spinup[-1]}_fdom{fdom}_k{k_label}.h5'

## [subsurface transport] - [left_subsurface bc]
print("modifying [subsurface transport] - [left_subsurface bc]")
domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'left_subsurface bc', 'boundary mole fraction function', 'dof 2 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'left_subsurface bc', 'boundary mole fraction function', 'dof 5 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'left_subsurface bc', 'boundary mole fraction function', 'dof 7 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

## [subsurface transport] - [right_subsurface bc]
print("modifying [subsurface transport] - [right_subsurface bc]")
domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'right_subsurface bc', 'boundary mole fraction function', 'dof 2 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'right_subsurface bc', 'boundary mole fraction function', 'dof 5 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'right_subsurface bc', 'boundary mole fraction function', 'dof 7 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

## [surface transport] - [right_surface bc]
print("modifying [surface transport] - [right_surface bc]")
domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'surface transport', 'boundary conditions', 'mole fraction', 'right_surface bc', 'boundary mole fraction function', 'dof 2 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'surface transport', 'boundary conditions', 'mole fraction', 'right_surface bc', 'boundary mole fraction function', 'dof 5 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'surface transport', 'boundary conditions', 'mole fraction', 'right_surface bc', 'boundary mole fraction function', 'dof 7 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

## revise other configurations specifically for casecybernetic

In [ ]:
## cycle driver
cycle_driver = asearch.find_path(xml, ['cycle driver', 'end time'])
cycle_driver.set("value", str(nyears_cyclic_spinup*365))
aio.toFile(xml, xml_filename)

cycle_driver = asearch.find_path(xml, ['cycle driver', 'end cycle'])
cycle_driver.set("value", "10000000") # 100,000 -> 166.4day, so for 3650day, change to 1e7
aio.toFile(xml, xml_filename)

In [ ]:
## visualization
visualization = asearch.find_path(xml, ['visualization', 'domain', 'times start period stop'])
visualization.set("value", "{0.0, 1.0, -1.0}")
aio.toFile(xml, xml_filename)

visualization = asearch.find_path(xml, ['visualization', 'domain', 'time units'])
visualization.set("value", "d")
aio.toFile(xml, xml_filename)

visualization = asearch.find_path(xml, ['visualization', 'surface', 'times start period stop'])
visualization.set("value", "{0.0, 1.0, -1.0}")
aio.toFile(xml, xml_filename)

visualization = asearch.find_path(xml, ['visualization', 'surface', 'time units'])
visualization.set("value", "d")
aio.toFile(xml, xml_filename)

In [ ]:
## checkpoint
checkpoint = asearch.find_path(xml, ['checkpoint'])
pl1a = Parameter(name="times start period stop", ptype="Array(double)", value="{0, 1, -1}")
pl1b = Parameter(name="times start period stop units", ptype="string", value="y")
pl1c = Parameter(name="file name digits", ptype="int", value="7") #should be enough for 10y sim.
checkpoint.append(pl1a)
checkpoint.append(pl1b)
checkpoint.append(pl1c)
aio.toFile(xml, xml_filename)

## revise initial conditions set in file pflotran_chemistry_cybernetic_smoothstep_pyc.txt
- this is only for the cybernetic spinup run1

In [ ]:
## load dynamic boundary conditions for NH4+, NO3-, and DOC from ELMimport h5py
hdf5_path = f'../data-processed/{site_name}/{site_name}_CNbc_conc_spinup_{nyears_cyclic_spinup}yr_{years_spinup[0]}_{years_spinup[-1]}_fdom{fdom}_k{k_label}.h5'
with h5.File(hdf5_path, "r") as f:
    time_spinup_bc = f['Time'][:]
    nh4_conc_spinup_bc = f['NH4+ mol water basis [molS molH^-1]'][:]
    no3_conc_spinup_bc = f['NO3- mol water basis [molS molH^-1]'][:]
    doc_conc_spinup_bc = f['DOC mol water basis v1 [molS molH^-1]'][:]

In [ ]:
## plot and calculate averaged values
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axs[0].plot(time_spinup_bc, nh4_conc_spinup_bc, color='b')
axs[0].set_ylabel('NH4+ [molS/molH]')
axs[0].set_title('NH4+ Concentration Over Time')

axs[1].plot(time_spinup_bc, no3_conc_spinup_bc, color='g')
axs[1].set_ylabel('NO3- [molS/molH]')
axs[1].set_title('NO3- Concentration Over Time')

axs[2].plot(time_spinup_bc, doc_conc_spinup_bc, color='r')
axs[2].set_ylabel('DOC [molS/molH]')
axs[2].set_title('DOC Concentration Over Time')
axs[2].set_xlabel('Time [days]')

plt.tight_layout()
plt.show()

In [ ]:
## calculate the average values
avg_nh4 = nh4_conc_spinup_bc.mean()
avg_no3 = no3_conc_spinup_bc.mean()
avg_doc = doc_conc_spinup_bc.mean()

## noticing that the concentration unit in pflotran_chemistry_cybernetic_smoothstep_pyc.txt is in molS/L
## so we need to convert the unit from molS/molH to molS/L
rho_m = 55000. # molH/m3, water molar density
avg_nh4_molS_L = avg_nh4 * rho_m / 1000  # convert from molS/molH to molS/L
avg_no3_molS_L = avg_no3 * rho_m / 1000  # convert from molS/molH to molS/L
avg_doc_molS_L = avg_doc * rho_m / 1000  # convert from molS/molH to molS/L

print(f"Time-averaged NH4+ concentration: {avg_nh4_molS_L:.3e} [molS/L]")
print(f"Time-averaged NO3- concentration: {avg_no3_molS_L:.3e} [molS/L]")
print(f"Time-averaged DOC concentration: {avg_doc_molS_L:.3e} [molS/L]")

## manual config
avg_doc_molS_L = 2.0e-4

In [ ]:
## revise file pflotran_chemistry_cybernetic_smoothstep_pyc.txt
## for cybernetic spinup run1
def replace_species(text, constraint_name, replacements):
    pattern = re.compile(
        rf"(CONSTRAINT {constraint_name}\s+CONCENTRATIONS\s+)(.*?)(/)",
        re.DOTALL
    )
    def repl(match):
        block = match.group(2)
        for species, value in replacements.items():
            block = re.sub(
                rf"({re.escape(species)}\s+)[\deE\.\-]+",
                lambda m: m.group(1) + f"{value:.5e}",
                block
            )
        return match.group(1) + block + match.group(3)
    return pattern.sub(repl, text)

file_path = target_folder_run1 /'reactions/pflotran_chemistry_cybernetic_smoothstep_pyc.txt'

replacements = {
    "CH2O(aq)": avg_doc_molS_L,
    "NH4+": avg_nh4_molS_L,
    "NO3-": avg_no3_molS_L
}

with open(file_path, "r") as f:
    content = f.read()

content = replace_species(content, "ICsubsurface", replacements)
content = replace_species(content, "ICsurface", replacements)

with open(file_path, "w") as f:
    f.write(content)


# modify the xml file for casecybernetic-run2

In [ ]:
# copy ats-pflotran xml for run1 to run2
source_path_run2 = base_dir / f'casecybernetic-run1.{flag_scenario}' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run1.v1.6_pflotran.xml'
target_folder_run2 = base_dir / f'casecybernetic-run2.{flag_scenario}'
target_path_run2 = target_folder_run2 / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run2.v1.6_pflotran.xml'
shutil.copy(source_path_run2, target_path_run2)

In [ ]:
xml_filename = base_dir / f'casecybernetic-run2.{flag_scenario}' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run2.v1.6_pflotran.xml'
xml = aio.fromFile(xml_filename)

## ~~revise the initial condition for flow part~~

## ~~revise the soil domain info used for DOC injection source terms~~

## revise the water_head/Daymet/MODIS_LAI hdf5 file info for flow part

In [ ]:
## water_head
outputs['BChead_start_merge_spinup_transient_filename_site'] = f'../data-processed/{site_name}/startpt_head_merged_spinup{nyears_cyclic_spinup}yr_transient.h5'
outputs['BChead_end_merge_spinup_transient_filename_site'] = f'../data-processed/{site_name}/endpt_head_merged_spinup{nyears_cyclic_spinup}yr_transient.h5'

## [subsurface transport] - [boundary conditions]
print("modifying [subsurface flow] - [boundary conditions]")
domain_subsrfflow_bc = asearch.find_path(xml, ['PKs', 'subsurface flow', 'boundary conditions', 'head', 'uphill', 'boundary head', 'function-tabular', 'file'])
domain_subsrfflow_bc.set("value", "../" + outputs['BChead_start_merge_spinup_transient_filename_site'])
print(domain_subsrfflow_bc)
aio.toFile(xml, xml_filename)

domain_subsrfflow_bc = asearch.find_path(xml, ['PKs', 'subsurface flow', 'boundary conditions', 'head', 'outlet', 'boundary head', 'function-tabular', 'file'])
domain_subsrfflow_bc.set("value", "../" + outputs['BChead_end_merge_spinup_transient_filename_site'])
print(domain_subsrfflow_bc)
aio.toFile(xml, xml_filename)

## [surface transport] - [boundary conditions]
print("modifying [surface flow] - [boundary conditions]")
domain_srfflow_bc = asearch.find_path(xml, ['PKs', 'surface flow', 'boundary conditions', 'head', 'uphill', 'boundary head', 'function-tabular', 'file'])
domain_srfflow_bc.set("value", "../" + outputs['BChead_start_merge_spinup_transient_filename_site'])
print(domain_srfflow_bc)
aio.toFile(xml, xml_filename)

domain_srfflow_bc = asearch.find_path(xml, ['PKs', 'surface flow', 'boundary conditions', 'head', 'outlet', 'boundary head', 'function-tabular', 'file'])
domain_srfflow_bc.set("value", "../" + outputs['BChead_end_merge_spinup_transient_filename_site'])
print(domain_srfflow_bc)
aio.toFile(xml, xml_filename)


In [ ]:
## daymet hdf5 file
outputs['daymet_merge_filename_site'] = f'../data-processed/{site_name}/{site_name}_DayMet_merged.h5'

print("modifying [state][evaluators][surface-incoming_shortwave_radiation]")
state_daymet = asearch.find_path(xml, ['state', 'evaluators', 'surface-incoming_shortwave_radiation', 'function', 'surface domain', 'function', 'file'])
state_daymet.set("value", "../" + outputs['daymet_merge_filename_site'])
print(state_daymet)
aio.toFile(xml, xml_filename)

print("modifying [state][evaluators][surface-precipitation_rain]")
state_daymet = asearch.find_path(xml, ['state', 'evaluators', 'surface-precipitation_rain', 'function', 'surface domain', 'function', 'file'])
state_daymet.set("value", "../" + outputs['daymet_merge_filename_site'])
print(state_daymet)
aio.toFile(xml, xml_filename)

print("modifying [state][evaluators][snow-precipitation]")
state_daymet = asearch.find_path(xml, ['state', 'evaluators', 'snow-precipitation', 'function', 'surface domain', 'function', 'file'])
state_daymet.set("value", "../" + outputs['daymet_merge_filename_site'])
print(state_daymet)
aio.toFile(xml, xml_filename)

print("modifying [state][evaluators][surface-vapor_pressure_air]")
state_daymet = asearch.find_path(xml, ['state', 'evaluators', 'surface-vapor_pressure_air', 'function', 'surface domain', 'function', 'file'])
state_daymet.set("value", "../" + outputs['daymet_merge_filename_site'])
print(state_daymet)
aio.toFile(xml, xml_filename)

print("modifying [state][evaluators][surface-air_temperature]")
state_daymet = asearch.find_path(xml, ['state', 'evaluators', 'surface-air_temperature', 'function', 'surface domain', 'function', 'file'])
state_daymet.set("value", "../" + outputs['daymet_merge_filename_site'])
print(state_daymet)
aio.toFile(xml, xml_filename)

print("modifying [state][evaluators][surface-temperature]")
state_daymet = asearch.find_path(xml, ['state', 'evaluators', 'surface-temperature', 'function', 'surface domain', 'function', 'file'])
state_daymet.set("value", "../" + outputs['daymet_merge_filename_site'])
print(state_daymet)
aio.toFile(xml, xml_filename)

In [ ]:
## MODIS_LAI hdf5 file
## first, load nlcd_labels
outputs['nlcd_labels_string'] = f'../data-processed/{site_name}/nlcd_labels.txt'
with open(outputs['nlcd_labels_string'], 'r') as f:
    nlcd_labels = f.read().splitlines()
print(nlcd_labels)

# Filter out 'Other' first
labels_to_process = [label for label in nlcd_labels if label != 'Other']

In [ ]:
outputs['modis_merged_filename_watershed_smoothed'] = f'../data-processed/{site_name}/{site_name}_MODIS_LAI_merged_smoothed.h5'

for label in labels_to_process:
    print(f"modifying [state][evaluators][canopy-leaf_area_index][function][{label}]")
    state_modislai = asearch.find_path(xml, ['state', 'evaluators', 'canopy-leaf_area_index', 'function', label, 'function', 'file'])
    state_modislai.set("value", "../" + outputs['modis_merged_filename_watershed_smoothed'])
    print(state_modislai)

aio.toFile(xml, xml_filename)

## revise the DOC injection hdf5 file info

In [ ]:
elm_filename = f'../../data-processed/{site_name}/{site_name}_DOC_source_merged_spinup{nyears_cyclic_spinup}yr_transient_{start_year_transient}_{end_year_transient}_fdom{fdom}_{run}.h5'

subsurface_mass_source = asearch.find_path(xml, ['PKs', 'source terms', 'component mass source', 'DOC production', 'source function', 'file'])
subsurface_mass_source.set("value", elm_filename)
print(subsurface_mass_source)
aio.toFile(xml, xml_filename)

## revise the CN boundary conditions hdf5 file info

In [ ]:
elm_filename = f'../../data-processed/{site_name}/{site_name}_CNbc_conc_merged_spinup{nyears_cyclic_spinup}yr_transient_{start_year_transient}_{end_year_transient}_fdom{fdom}_{run}.h5'

## [subsurface transport] - [left_subsurface bc]
print("modifying [subsurface transport] - [left_subsurface bc]")
domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'left_subsurface bc', 'boundary mole fraction function', 'dof 2 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'left_subsurface bc', 'boundary mole fraction function', 'dof 5 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'left_subsurface bc', 'boundary mole fraction function', 'dof 7 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

## [subsurface transport] - [right_subsurface bc]
print("modifying [subsurface transport] - [right_subsurface bc]")
domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'right_subsurface bc', 'boundary mole fraction function', 'dof 2 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'right_subsurface bc', 'boundary mole fraction function', 'dof 5 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'subsurface transport', 'boundary conditions', 'mole fraction', 'right_subsurface bc', 'boundary mole fraction function', 'dof 7 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

## [surface transport] - [right_surface bc]
print("modifying [surface transport] - [right_surface bc]")
domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'surface transport', 'boundary conditions', 'mole fraction', 'right_surface bc', 'boundary mole fraction function', 'dof 2 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'surface transport', 'boundary conditions', 'mole fraction', 'right_surface bc', 'boundary mole fraction function', 'dof 5 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

domain_CNbc_conc = asearch.find_path(xml, ['PKs', 'surface transport', 'boundary conditions', 'mole fraction', 'right_surface bc', 'boundary mole fraction function', 'dof 7 function', 'function-tabular', 'file'])
domain_CNbc_conc.set("value", elm_filename)
print(domain_CNbc_conc)
aio.toFile(xml, xml_filename)

## revise other configurations specifically for casecybernetic

In [ ]:
## cycle driver
cycle_driver = asearch.find_path(xml, ['cycle driver', 'end time'])
total_years = nyears_cyclic_spinup + (end_year_transient - start_year_transient + 1)
cycle_driver.set("value", str(total_years*365))
aio.toFile(xml, xml_filename)

cycle_driver = asearch.find_path(xml, ['cycle driver', 'end cycle'])
cycle_driver.set("value", "20000000") # 100,000 -> 166.4day, so for 3650day, change to 1e7
aio.toFile(xml, xml_filename)

## ~~revise initial conditions set in file pflotran_chemistry_cybernetic_smoothstep_pyc.txt~~